In [113]:
import os
import geopandas as gpd
import folium
from shapely.geometry import Polygon, Point
import json

In [114]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [115]:
filepath = "/content/drive/MyDrive/Data Hackathon_IA_2025/couche/"

# Liste des fichiers .geojson
geojson_files = [file for file in os.listdir(filepath) if file.endswith(".geojson")]

geo_layers = {}
for file in geojson_files:
    full_path = os.path.join(filepath, file)
    geo_layers[file.replace(".geojson", "")] = gpd.read_file(full_path)

In [116]:


def convert_datetime_columns_to_str(gdf):
    datetime_cols = gdf.select_dtypes(include=['datetime64[ns]', 'datetime64[ns, UTC]']).columns
    for col in datetime_cols:
        gdf[col] = gdf[col].astype(str)
    return gdf

In [117]:
andf_layers = ['aif', 'air_proteges',
              'dpl', 'dpm', 'enregistrement individuel',
              'litige', 'parcelles', 'restriction', 'tf_demembres',
              'tf_en_cours', 'tf_etat', 'titre_reconstitue', 'zone_inondable']

# Filtrage des couches existantes
andf_layers = {layer_name: layer for layer_name, layer in geo_layers.items() if layer_name in andf_layers}

In [124]:
from shapely.geometry import Polygon
import geopandas as gpd

def process_coordinates(raw_coords_dict: list, source_crs: str):
    """
    Crée le polygone du terrain à partir des coordonnées brutes (liste de dicts)
    et le reprojecte en WGS 84.

    Args:
        raw_coords_dict (list): Liste des dictionnaires de coordonnées [{"x": X, "y": Y}].
        source_crs (str): Le code EPSG du CRS source (ex: 'EPSG:32231').

    Returns:
        tuple: (
            parcel: Polgone du terrain dans le CRS source (pour le filtrage),
            parcel_wgs84: Polgone du terrain en WGS 84 (pour Folium),
            center_lon: Longitude du centre en WGS 84,
            center_lat: Latitude du centre en WGS 84
        )
    """
    # 1. Conversion de la liste de dictionnaires en liste de tuples (X, Y)
    points = [(c["x"], c["y"]) for c in raw_coords_dict]

    # 2. Création du polygone dans le CRS source
    parcel = Polygon(points)
    parcel_gdf = gpd.GeoDataFrame(index=[0], crs=source_crs, geometry=[parcel])

    # 3. Reprojection en WGS 84 (EPSG:4326)
    parcel_wgs84_gdf = parcel_gdf.to_crs(epsg=4326)
    parcel_wgs84 = parcel_wgs84_gdf.iloc[0].geometry

    # 4. Calcul du centroïde en WGS 84
    center_lon = parcel_wgs84.centroid.x
    center_lat = parcel_wgs84.centroid.y

    return parcel, parcel_wgs84, center_lon, center_lat



In [121]:
layer_colors = {
    "aif": "#006400",
    "air_proteges": "#228B22",
    "dpl": "#1E90FF",
    "dpm": "#00CED1",
    "enregistrement individuel": "#FFD700",
    "litige": "#FF0000",
    "parcelles": "#8B4513",
    "restriction": "#FF8C00",
    "tf_demembres": "#9932CC",
    "tf_en_cours": "#00FF7F",
    "tf_etat": "#4169E1",
    "titre_reconstitue": "#DC143C",
    "zone_inondable": "#87CEFA"
}


In [125]:

def filter_layer_by_distance(gdf, center_point, max_dist):

    return gdf[gdf.geometry.distance(center_point) <= max_dist]


center_point = parcel.centroid
radius_meters = 5000


for layer_name, gdf in andf_layers.items():


    if gdf.crs is None:
        gdf.set_crs(SCR_SOURCE, inplace=True)


    filtered_gdf = filter_layer_by_distance(gdf, center_point, radius_meters)

    geojson_data = None


    if filtered_gdf.empty:

        geojson_data = {"type": "FeatureCollection", "features": []}
    else:

        filtered_gdf_wgs84 = filtered_gdf.to_crs(epsg=4326)

        filtered_gdf_str_converted = convert_datetime_columns_to_str(filtered_gdf_wgs84)
        geojson_str = filtered_gdf_str_converted.to_json()
        geojson_data = json.loads(geojson_str)

    color = layer_colors.get(layer_name, "gray")


    popup_fields = None

    if layer_name == "tf_demembres":

        popup_fields = folium.GeoJsonPopup(
            fields=['num_tf'],
            aliases=['Numéro du Titre Foncier Démembré:'],
            localize=True
        )
    elif layer_name == "tf_en_cours":

        popup_fields = folium.GeoJsonPopup(
            fields=['validation', 'motif'],
            aliases=['Validation (Oui/Non):', 'Motif (si Non):'],
            localize=True
        )
    elif layer_name == "litige":

        popup_fields = folium.GeoJsonPopup(
            fields=['obs'],
            aliases=['Observations sur le Litige:'],
            localize=True
        )
    elif layer_name == "restriction":

        popup_fields = folium.GeoJsonPopup(
            fields=['type', 'designation'],
            aliases=['Type de Restriction:', 'Détails:'],
            localize=True
        )


    folium.GeoJson(
        geojson_data,
        name=layer_name,
        popup=popup_fields,
        style_function=lambda feature, col=color: {
            'fillColor': col,
            'color': col,
            'weight': 2,
            'fillOpacity': 0.4
        }
    ).add_to(m)

points_wgs84_folium = [(lat, lon) for lon, lat in parcel_wgs84.exterior.coords]

folium.Polygon(
    locations=points_wgs84_folium,
    color='blue',
    weight=3,
    fill=True,
    fill_opacity=0.2,
    popup='Terrain utilisateur'
).add_to(m)


folium.LayerControl().add_to(m)

In [ ]:
m